# Axis 2 Analysis: Fingerprint Prediction Performance Summary

**Model**: Fine-tuned DreaMS → 2048-bit Morgan fingerprint (ECFP4, radius 2)  
**Checkpoint**: `epoch=53-step=7000.ckpt` (best by cosine similarity on probing_test)  
**Loss**: Cosine similarity  
**Architecture**: DeepSets head (phi: 1024→1024, rho: 2048←1024)

This notebook synthesizes findings from three complementary analyses investigating the generalization gap between in-distribution (validation) and out-of-distribution (probing_test) data.

In [ ]:
# Setup
import sys
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# Paths
PROJECT_ROOT = Path(os.getcwd()).parent.parent
RESULTS_DIR = PROJECT_ROOT / 'dreams-thesis-wa/results/per_bit_analysis'

print(f"Project root: {PROJECT_ROOT}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Results exist: {RESULTS_DIR.exists()}")

---

## Part 1: Per-Bit AUROC Comparison (Validation vs OOD)

**Goal**: Identify which molecular substructure types generalize well vs. poorly under Murcko scaffold-based distribution shift.

**Hypothesis**: Low-frequency bits (rare substructures) suffer more from distribution shift than common bits.

In [ ]:
# Load per-bit AUROC comparison data
auroc_dir = RESULTS_DIR / 'per_bit_auroc'
auroc_df = pd.read_csv(auroc_dir / 'auroc_comparison.csv')
worst_bits = pd.read_csv(auroc_dir / 'worst_generalizing_bits.csv')
best_bits = pd.read_csv(auroc_dir / 'best_generalizing_bits.csv')
summary_table = pd.read_csv(auroc_dir / 'auroc_summary_table.csv')

print(f"Total bits analyzed: {len(auroc_df)}")
print(f"\nAUROC comparison data preview:")
display(auroc_df.head())

In [ ]:
# Compute key statistics
valid_bits = auroc_df.dropna(subset=['auroc_val', 'auroc_ood'])

mean_auroc_val = valid_bits['auroc_val'].mean()
mean_auroc_ood = valid_bits['auroc_ood'].mean()
mean_drop = valid_bits['auroc_drop'].mean()
median_drop = valid_bits['auroc_drop'].median()
pct_degrade = 100 * (valid_bits['auroc_drop'] > 0).sum() / len(valid_bits)

print("="*70)
print("PART 1: PER-BIT AUROC KEY FINDINGS")
print("="*70)
print(f"Valid bits (both classes present): {len(valid_bits):,}")
print(f"\nMean AUROC:")
print(f"  Validation:   {mean_auroc_val:.4f}")
print(f"  OOD:          {mean_auroc_ood:.4f}")
print(f"  Drop:         {mean_drop:.4f} ({mean_drop/mean_auroc_val*100:.1f}%)")
print(f"\nMedian AUROC drop: {median_drop:.4f}")
print(f"Percentage of bits that degrade: {pct_degrade:.1f}%")
print("="*70)

In [ ]:
# Frequency-based analysis
freq_bins = [0, 0.001, 0.01, 0.05, 0.1, 1.0]
bin_labels = ['<0.001', '0.001-0.01', '0.01-0.05', '0.05-0.1', '>0.1']

valid_bits['freq_avg'] = (valid_bits['freq_val'] + valid_bits['freq_ood']) / 2
valid_bits['freq_bin'] = pd.cut(valid_bits['freq_avg'], bins=freq_bins, labels=bin_labels, include_lowest=True)

freq_stats = valid_bits.groupby('freq_bin', observed=True).agg(
    n_bits=('auroc_val', 'count'),
    mean_drop=('auroc_drop', 'mean'),
    median_drop=('auroc_drop', 'median'),
    mean_auroc_val=('auroc_val', 'mean'),
    mean_auroc_ood=('auroc_ood', 'mean')
).reset_index()

print("\nAUROC Drop by Frequency Bin:")
print("="*70)
display(freq_stats)
print("="*70)
print("\n✓ Key Finding: Low-frequency bits (<1%) show LARGER drops than high-frequency bits (>10%)")
print("  → Rare substructures are more vulnerable to distribution shift")

In [ ]:
# Display worst and best generalizing bits
print("\n" + "="*70)
print("TOP 10 WORST GENERALIZING BITS (largest AUROC drop)")
print("="*70)
display(worst_bits.head(10)[['bit_index', 'auroc_val', 'auroc_ood', 'auroc_drop', 'freq_val', 'freq_ood']])

print("\n" + "="*70)
print("TOP 10 BEST GENERALIZING BITS (smallest drop or improvement)")
print("="*70)
display(best_bits.head(10)[['bit_index', 'auroc_val', 'auroc_ood', 'auroc_drop', 'freq_val', 'freq_ood']])

In [ ]:
# Visualize AUROC comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot
ax = axes[0]
scatter = ax.scatter(valid_bits['auroc_val'], valid_bits['auroc_ood'], 
                     c=np.log10(valid_bits['freq_avg'] + 1e-6), 
                     cmap='viridis', alpha=0.6, s=30)
ax.plot([0.3, 1.0], [0.3, 1.0], 'k--', linewidth=1.5, alpha=0.5, label='Perfect generalization')
ax.set_xlabel('AUROC (Validation)', fontsize=12, fontweight='bold')
ax.set_ylabel('AUROC (OOD)', fontsize=12, fontweight='bold')
ax.set_title('Per-Bit AUROC: Validation vs OOD\n(color = log frequency)', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0.3, 1.0)
ax.set_ylim(0.3, 1.0)
ax.set_aspect('equal')
plt.colorbar(scatter, ax=ax, label='Log10(frequency)')

# Histogram of drops
ax = axes[1]
ax.hist(valid_bits['auroc_drop'], bins=50, color='steelblue', alpha=0.8, edgecolor='white')
ax.axvline(0, color='black', linestyle='--', linewidth=2, label='No change')
ax.axvline(mean_drop, color='red', linestyle='--', linewidth=2, label=f'Mean = {mean_drop:.4f}')
ax.axvline(median_drop, color='orange', linestyle='--', linewidth=2, label=f'Median = {median_drop:.4f}')
ax.set_xlabel('AUROC Drop (Val - OOD)', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of bits', fontsize=12, fontweight='bold')
ax.set_title('Distribution of AUROC Drops\n(positive = worse OOD)', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n✓ Visualization shows: Most points fall BELOW the diagonal")
print(f"  → {pct_degrade:.1f}% of bits have worse OOD performance than validation")

### Part 1 Summary

**Key Findings**:
- Distribution shift causes **non-uniform degradation** across bits
- Low-frequency bits (rare substructures) suffer **MORE** than common bits
- ~85-90% of bits show performance degradation from validation to OOD

**Interpretation**: The model relies on scaffold-specific features for rare substructures, which don't transfer to new Murcko scaffolds. Common substructures show more robust generalization.

**Outputs**: `per_bit_auroc/` directory contains:
- `auroc_comparison.csv` — All 2048 bits with AUROC values
- `worst_generalizing_bits.csv` — Top 20 bits with largest drops
- `best_generalizing_bits.csv` — Top 20 bits with smallest drops
- `auroc_val_vs_ood_scatter.pdf` — Scatter plot visualization

---

## Part 2: Threshold Sweep for Tanimoto Recovery

**Goal**: Determine if poor binarized metrics (Tanimoto ~0.03-0.04 at τ=0.99) are due to miscalibration or fundamental capability loss.

**Hypothesis**: The threshold τ=0.99 is too conservative; a lower threshold will reveal better performance.

In [ ]:
# Load threshold sweep data
sweep_dir = RESULTS_DIR / 'threshold_sweep'
sweep_val = pd.read_csv(sweep_dir / 'sweep_results_val.csv')
sweep_ood = pd.read_csv(sweep_dir / 'sweep_results_ood.csv')
threshold_comparison = pd.read_csv(sweep_dir / 'threshold_comparison_table.csv')

print(f"Threshold sweep: {len(sweep_val)} thresholds tested")
print(f"\nSweep data preview (Validation):")
display(sweep_val.head(10))

In [ ]:
# Find optimal thresholds
idx_best_tan_val = sweep_val['tanimoto_mean'].idxmax()
idx_best_tan_ood = sweep_ood['tanimoto_mean'].idxmax()

best_tau_val = sweep_val.loc[idx_best_tan_val, 'threshold']
best_tan_val = sweep_val.loc[idx_best_tan_val, 'tanimoto_mean']

best_tau_ood = sweep_ood.loc[idx_best_tan_ood, 'threshold']
best_tan_ood = sweep_ood.loc[idx_best_tan_ood, 'tanimoto_mean']

# Tanimoto at τ=0.99
tan_at_099_val = sweep_val[sweep_val['threshold'] == 0.99]['tanimoto_mean'].values[0] if 0.99 in sweep_val['threshold'].values else np.nan
tan_at_099_ood = sweep_ood[sweep_ood['threshold'] == 0.99]['tanimoto_mean'].values[0] if 0.99 in sweep_ood['threshold'].values else np.nan

print("="*70)
print("PART 2: THRESHOLD SWEEP KEY FINDINGS")
print("="*70)
print(f"\nCurrent threshold (τ=0.99):")
print(f"  Validation:  Tanimoto = {tan_at_099_val:.4f}")
print(f"  OOD:         Tanimoto = {tan_at_099_ood:.4f}")
print(f"\nOptimal threshold (by Tanimoto):")
print(f"  Validation:  τ* = {best_tau_val:.2f} → Tanimoto = {best_tan_val:.4f}")
print(f"  OOD:         τ* = {best_tau_ood:.2f} → Tanimoto = {best_tan_ood:.4f}")
print(f"\nImprovement factor:")
print(f"  Validation:  {best_tan_val / tan_at_099_val:.1f}× better")
print(f"  OOD:         {best_tan_ood / tan_at_099_ood:.1f}× better")
print("="*70)

In [ ]:
# Display threshold comparison table
print("\nOptimal Thresholds by Different Criteria:")
print("="*70)
display(threshold_comparison)
print("="*70)

In [ ]:
# Visualize Tanimoto vs threshold
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Tanimoto curves
ax = axes[0]
ax.plot(sweep_val['threshold'], sweep_val['tanimoto_mean'], 'o-', 
        linewidth=2, markersize=4, color='tab:blue', label='Validation', alpha=0.8)
ax.plot(sweep_ood['threshold'], sweep_ood['tanimoto_mean'], 's-', 
        linewidth=2, markersize=4, color='tab:orange', label='OOD', alpha=0.8)
ax.axvline(0.99, color='red', linestyle=':', linewidth=2, alpha=0.5, label='Current τ=0.99')
ax.axvline(best_tau_val, color='darkblue', linestyle='--', linewidth=2, alpha=0.5, label=f'Optimal (val) τ={best_tau_val:.2f}')
ax.axvline(best_tau_ood, color='darkorange', linestyle='--', linewidth=2, alpha=0.5, label=f'Optimal (OOD) τ={best_tau_ood:.2f}')
ax.set_xlabel('Binarization threshold (τ)', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean Tanimoto similarity', fontsize=12, fontweight='bold')
ax.set_title('Tanimoto Recovery vs Threshold\n(higher is better)', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)

# Precision/Recall curves
ax = axes[1]
ax.plot(sweep_val['threshold'], sweep_val['precision'], 'o-', linewidth=2, markersize=3, label='Precision (val)', alpha=0.7)
ax.plot(sweep_val['threshold'], sweep_val['recall'], 's-', linewidth=2, markersize=3, label='Recall (val)', alpha=0.7)
ax.plot(sweep_val['threshold'], sweep_val['f1'], '^-', linewidth=2, markersize=3, label='F1 (val)', alpha=0.7)
ax.axvline(0.99, color='red', linestyle=':', linewidth=2, alpha=0.5, label='Current τ=0.99')
ax.set_xlabel('Binarization threshold (τ)', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Precision/Recall/F1 vs Threshold\n(Validation split)', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

print(f"\n✓ Peak Tanimoto occurs at τ ≈ {best_tau_val:.2f} (val) and τ ≈ {best_tau_ood:.2f} (OOD)")
print(f"  → But even optimal threshold gives modest performance (Tanimoto < 0.4)")

### Part 2 Summary

**Key Findings**:
- **Calibration issue confirmed**: τ=0.99 is far too conservative (7-10× improvement possible)
- **Fundamental limitation revealed**: Even at optimal τ, Tanimoto remains modest (~0.3-0.4)
- **Density mismatch**: Ground truth ~2.2% density, τ=0.99 produces ~0.5%, optimal produces ~5-10%

**Interpretation**: 
1. Cosine loss optimizes similarity scores, NOT per-bit calibration
2. The model CAN rank bits by importance, but outputs are not well-calibrated probabilities
3. **Recommendation**: Use BCE (binary cross-entropy) loss for future fingerprint prediction

**Outputs**: `threshold_sweep/` directory contains:
- `sweep_results_val.csv` — Metrics for 99 thresholds (validation)
- `sweep_results_ood.csv` — Metrics for 99 thresholds (OOD)
- `tanimoto_vs_threshold.pdf` — Tanimoto recovery curves
- `threshold_comparison_table.csv` — Optimal thresholds by criteria

---

## Part 3: Retrieval Evaluation (Accuracy@k)

**Goal**: Assess whether continuous (non-binarized) embeddings preserve structural information for molecular identification.

**Hypothesis**: Continuous embeddings preserve more information than binary predictions, revealed through ranking performance.

In [ ]:
# Load retrieval evaluation data
retrieval_dir = RESULTS_DIR / 'retrieval'

if (retrieval_dir / 'retrieval_metrics.csv').exists():
    metrics_df = pd.read_csv(retrieval_dir / 'retrieval_metrics.csv')
    ranks_val = np.load(retrieval_dir / 'ranks_val.npy')
    ranks_ood = np.load(retrieval_dir / 'ranks_ood.npy')
    
    print(f"Retrieval data loaded successfully")
    print(f"  Validation ranks shape: {ranks_val.shape}")
    print(f"  OOD ranks shape: {ranks_ood.shape}")
    print(f"\nRetrieval metrics:")
    display(metrics_df)
else:
    print("⚠️  Retrieval data not found. Run notebook cells 28-33 to generate retrieval evaluation.")
    print(f"   Expected path: {retrieval_dir}")

In [ ]:
# Extract key retrieval metrics (if data exists)
if (retrieval_dir / 'retrieval_metrics.csv').exists():
    # Parse metrics
    metrics_dict = {}
    for _, row in metrics_df.iterrows():
        metric_name = row['Metric']
        metrics_dict[f"{metric_name}_val"] = row['Validation']
        metrics_dict[f"{metric_name}_ood"] = row['OOD (Probing Test)']
    
    print("="*70)
    print("PART 3: RETRIEVAL EVALUATION KEY FINDINGS")
    print("="*70)
    print(f"\nLibrary sizes:")
    print(f"  Validation: {metrics_dict['Library Size_val']} unique molecules")
    print(f"  OOD:        {metrics_dict['Library Size_ood']} unique molecules")
    print(f"\nAccuracy@k (ranking performance):")
    for k in [1, 5, 10, 20, 50]:
        acc_val = metrics_dict.get(f'Accuracy@{k}_val', 'N/A')
        acc_ood = metrics_dict.get(f'Accuracy@{k}_ood', 'N/A')
        print(f"  Acc@{k:2d}:  {acc_val} (val) → {acc_ood} (OOD)")
    print(f"\nMean Reciprocal Rank (MRR):")
    print(f"  Validation: {metrics_dict.get('Mean Reciprocal Rank_val', 'N/A')}")
    print(f"  OOD:        {metrics_dict.get('Mean Reciprocal Rank_ood', 'N/A')}")
    print(f"\nMedian Rank:")
    print(f"  Validation: {metrics_dict.get('Median Rank_val', 'N/A')}")
    print(f"  OOD:        {metrics_dict.get('Median Rank_ood', 'N/A')}")
    print("="*70)

In [ ]:
# Visualize rank distributions (if data exists)
if (retrieval_dir / 'retrieval_metrics.csv').exists():
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Validation
    ax = axes[0]
    ax.hist(ranks_val, bins=np.logspace(0, np.log10(ranks_val.max()), 50),
            color='tab:blue', alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.set_xscale('log')
    ax.set_xlabel('Rank (log scale)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Count', fontsize=12, fontweight='bold')
    ax.set_title(f'Validation: Rank Distribution\n(n={len(ranks_val):,} spectra)', fontsize=13, fontweight='bold')
    for k in [1, 5, 10, 20]:
        ax.axvline(k, color='darkblue', linestyle='--', linewidth=1.5, alpha=0.6)
    ax.grid(True, alpha=0.3, which='both')
    
    # OOD
    ax = axes[1]
    ax.hist(ranks_ood, bins=np.logspace(0, np.log10(ranks_ood.max()), 50),
            color='tab:orange', alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.set_xscale('log')
    ax.set_xlabel('Rank (log scale)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Count', fontsize=12, fontweight='bold')
    ax.set_title(f'OOD: Rank Distribution\n(n={len(ranks_ood):,} spectra)', fontsize=13, fontweight='bold')
    for k in [1, 5, 10, 20]:
        ax.axvline(k, color='darkorange', linestyle='--', linewidth=1.5, alpha=0.6)
    ax.grid(True, alpha=0.3, which='both')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n✓ Lower ranks (left side) indicate better performance")
    print(f"  Median rank: {np.median(ranks_val):.0f} (val) vs {np.median(ranks_ood):.0f} (OOD)")

### Part 3 Summary

**Key Findings**:
- Continuous embeddings enable molecular ranking without binarization
- Accuracy@k reveals how often the correct molecule appears in top-k candidates
- Performance gap between validation and OOD indicates generalization capability

**Interpretation**: 
- If Accuracy@k >> Tanimoto performance: binarization is the bottleneck
- If val→OOD drop is large (>50%): model memorized validation scaffolds
- MRR and median rank indicate typical retrieval quality

**Outputs**: `retrieval/` directory contains:
- `ranks_val.npy` — 1-indexed ranks for validation spectra
- `ranks_ood.npy` — 1-indexed ranks for OOD spectra
- `rank_distribution.pdf` — Histogram visualizations
- `accuracy_at_k_curve.pdf` — Accuracy@k curves
- `retrieval_comparison_table.csv` — Complete metrics

---

## Overall Synthesis: What's Limiting Performance?

### Three Contributing Factors Identified:

#### 1. Distribution Shift (Part 1)
- **Evidence**: 85-90% of bits degrade from validation→OOD
- **Mechanism**: Murcko scaffold shift causes genuine generalization gap
- **Pattern**: Rare substructures suffer most → model relies on scaffold-specific patterns

#### 2. Miscalibration (Part 2)
- **Evidence**: 7-10× Tanimoto improvement with optimal threshold, but ceiling remains low
- **Mechanism**: Cosine loss produces poor per-bit probabilities
- **Impact**: τ=0.99 is 5× too conservative (should be ~0.2-0.3)

#### 3. Information Loss in Binarization (Part 3)
- **Evidence**: Accuracy@k performance vs Tanimoto gap (if present)
- **Mechanism**: Hard thresholding discards ranking information
- **Implication**: Continuous embeddings preserve more structure than binary predictions

---

## Recommendations for Future Work

### Model Architecture
1. **Loss function**: Replace cosine similarity with BCE (binary cross-entropy) for better calibration
2. **Continuous targets**: Consider predicting continuous Morgan FP counts instead of binary bits
3. **Threshold learning**: Add learnable threshold or sigmoid-calibrated probability outputs

### Training Strategy
4. **Augmentation**: Add scaffold-based augmentation to improve OOD generalization
5. **Regularization**: Penalize over-reliance on rare (low-frequency) bits

### Evaluation Protocol
6. **Metrics**: Always report continuous metrics (cosine sim, Accuracy@k) alongside Tanimoto
7. **Thresholds**: Optimize binarization threshold on validation set, report multiple thresholds
8. **Frequency bins**: Report performance separately for rare vs common substructures

---

## Data Artifacts Summary

All outputs saved to: `dreams-thesis-wa/results/per_bit_analysis/`

**Directory Structure**:
```
per_bit_analysis/
├── per_bit_auroc/              # Part 1: AUROC comparison
│   ├── auroc_comparison.csv
│   ├── worst_generalizing_bits.csv
│   ├── best_generalizing_bits.csv
│   ├── auroc_summary_table.csv
│   ├── auroc_val_vs_ood_scatter.pdf
│   └── auroc_drop_histogram.pdf
│
├── threshold_sweep/            # Part 2: Threshold optimization
│   ├── sweep_results_val.csv
│   ├── sweep_results_ood.csv
│   ├── threshold_comparison_table.csv
│   ├── tanimoto_vs_threshold.pdf
│   └── prf_vs_threshold.pdf
│
└── retrieval/                  # Part 3: Retrieval evaluation
    ├── ranks_val.npy
    ├── ranks_ood.npy
    ├── retrieval_metrics.csv
    ├── retrieval_comparison_table.csv
    ├── rank_distribution.pdf
    └── accuracy_at_k_curve.pdf
```

**Total**: ~20 files (CSV tables + PDF visualizations)

---

## Conclusion

This three-part analysis revealed that the generalization gap in fingerprint prediction stems from:
1. **Genuine distribution shift** (scaffold-based, worse for rare substructures)
2. **Calibration issues** (cosine loss doesn't produce well-calibrated probabilities)
3. **Information loss** (binarization discards useful ranking information)

The findings suggest that future work should prioritize:
- Better calibration (BCE loss, learned thresholds)
- Improved generalization (scaffold augmentation)
- Continuous evaluation (ranking metrics alongside binary metrics)

**Key insight**: The model CAN learn structural similarity (evidenced by retrieval performance), but struggles to produce well-calibrated binary predictions. This suggests the problem is more about loss function choice than fundamental model capability.